 ================================
# 08 - RLHF-style Preference Tuning
 
 #### Using DPO (Direct Preference Optimization) with TRL
 ================================


### 📓 Notebook 08 — 08_rlhf_preference_tuning.ipynb

Goal of this notebook:
	•	Take your existing instruction → answer SFT dataset
	•	Simulate preferences: good vs bad answers
	•	Train a small RLHF-style model using DPO (Direct Preference Optimization) via trl
	•	This is much lighter than real PPO, ideal for your MacBook
	•	Save a new model: gpt2-sales-rlhf

🧩 This is “RLHF-style” — we’re not doing a full PPO loop (which is heavier), but we are doing preference-based training, which is what interviewers care about.

⸻

🔧 Assumptions

I’ll assume from your SFT notebook:
	•	You already have sales_llm_instructions.csv (or similar) somewhere like data/llm/sft_instructions.csv
	•	You used gpt2 or gpt2-medium as base
	•	You saved the SFT model here: SFT_MODEL_DIR = "outputs/llm/gpt2-sales-sft"

## 🧱 Cell 1 — Imports & basic setup

### First cell: set paths like this (important)

In [1]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

from pathlib import Path
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

SFT_MODEL_DIR = Path("../outputs/llm/gpt2-sales-sft")
RLHF_MODEL_DIR = Path("../outputs/llm/gpt2-sales-rlhf")

device = "mps" if torch.backends.mps.is_available() else "cpu"

Using device: mps


/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load tokenizer + models

In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained(str(SFT_MODEL_DIR))
tokenizer.pad_token = tokenizer.eos_token

ref_model = GPT2LMHeadModel.from_pretrained(str(SFT_MODEL_DIR)).to(device)
policy_model = GPT2LMHeadModel.from_pretrained(str(SFT_MODEL_DIR)).to(device)

ref_model.eval()
policy_model.train()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Run RLHF (DPO) training

When training completes, make sure you explicitly save:

In [3]:
# dpo_trainer.save_model(str(RLHF_MODEL_DIR))
# tokenizer.save_pretrained(str(RLHF_MODEL_DIR))

Then verify:

In [4]:
print("RLHF exists:", RLHF_MODEL_DIR.exists())
print(list(RLHF_MODEL_DIR.iterdir()))

RLHF exists: True
[]


In [5]:
from trl import DPOTrainer, DPOConfig
print("TRL imported!")

TRL imported!


In [6]:
import os

for root, dirs, files in os.walk("/Users/imvenu/turing-ai-project", topdown=True):
    for f in files:
        if "sales_llm" in f.lower():
            print(os.path.join(root, f))

/Users/imvenu/turing-ai-project/data/sft/sales_llm_instructions.csv
/Users/imvenu/turing-ai-project/data/sft/sales_llm_dataset.jsonl


In [7]:
os.listdir("/Users/imvenu/turing-ai-project/data")
os.listdir("/Users/imvenu/turing-ai-project/data/sft")

['sales_llm_instructions.csv', 'sales_llm_dataset.jsonl']

In [8]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed")

# List all processed files to see what exists
print("Available processed files:", list(processed_dir.iterdir()))

possible_files = [
    "df_with_features.csv",   # ✅ THIS is your actual file
    "daily_clean.csv",
    "cleaned_daily_sales.csv",
    "pharmacy_daily.csv",
    "daily_merged.csv",
    "daily_final.csv",
]

daily_path = None
for f in possible_files:
    if (processed_dir / f).exists():
        daily_path = processed_dir / f
        break

if daily_path is None:
    raise FileNotFoundError(
        "No valid processed dataset found.\nChecked:\n" +
        "\n".join(str(processed_dir / f) for f in possible_files)
    )

print("✔ Using sales dataset:", daily_path)

df_sales = pd.read_csv(daily_path, parse_dates=["datum"])
df_sales.head()

print(df_sales.columns)
df_sales.head()

Available processed files: [PosixPath('../data/processed/df_with_features.csv')]
✔ Using sales dataset: ../data/processed/df_with_features.csv
Index(['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03',
       'R06', 'Year', 'Month', 'Hour', 'Weekday Name', 'Day', 'Weekday',
       'Weekday_Name', 'Week', 'IsWeekend', 'N02BE_lag1', 'N02BE_lag7',
       'N02BE_lag30', 'N02BE_roll7', 'N02BE_roll30', 'N02BE_roll7_std',
       'N02BE_cummean', 'N02BE_cumsum', 'yearly_sin_1', 'yearly_cos_1',
       'yearly_sin_2', 'yearly_cos_2', 'yearly_sin_3', 'yearly_cos_3',
       'weekly_sin_1', 'weekly_cos_1', 'weekly_sin_2', 'weekly_cos_2',
       'weekly_sin_3', 'weekly_cos_3', 'Weekday_Avg', 'Month_Avg'],
      dtype='object')


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,...,yearly_sin_3,yearly_cos_3,weekly_sin_1,weekly_cos_1,weekly_sin_2,weekly_cos_2,weekly_sin_3,weekly_cos_3,Weekday_Avg,Month_Avg
0,2014-02-01,4.33,4.32,5.0,43.0,13.0,1.0,14.0,0.0,2014,...,0.999769,0.021516,0.974928,-0.222521,-0.433884,-0.900969,-0.781831,0.623490,33.579719,36.13891
1,2014-02-02,7.00,3.00,0.2,13.5,6.0,2.0,8.0,0.0,2014,...,0.999546,-0.030120,0.433884,-0.900969,-0.781831,0.623490,0.974928,-0.222521,33.392859,36.13891
2,2014-02-03,5.00,1.00,8.5,32.4,16.0,1.0,1.0,0.0,2014,...,0.996659,-0.081676,-0.433884,-0.900969,0.781831,0.623490,-0.974928,-0.222521,29.232601,36.13891
3,2014-02-04,1.33,3.00,7.0,30.6,8.0,1.0,17.0,2.0,2014,...,0.991114,-0.133015,-0.974928,-0.222521,0.433884,-0.900969,0.781831,0.623490,28.373665,36.13891
4,2014-02-05,3.00,4.02,6.2,32.4,15.0,1.0,1.0,1.0,2014,...,0.982927,-0.183998,-0.781831,0.623490,-0.974928,-0.222521,-0.433884,-0.900969,28.074514,36.13891


## 🧱 Cell 2 — Paths & folders

In [9]:
from pathlib import Path

# Base directory (your project root)
BASE_DIR = Path("..")  # notebook in /notebooks

# Data directories
DATA_DIR = BASE_DIR / "data"
LLM_DATA_DIR = DATA_DIR / "llm"

# Model directories
MODEL_DIR = BASE_DIR / "models" / "sft_gpt2"   # your real SFT model!
RLHF_MODEL_DIR = BASE_DIR / "models" / "rlhf_gpt2"  # new RLHF model

# SFT dataset path
SFT_DATA_PATH = DATA_DIR / "sft" / "sales_llm_instructions.csv"

# Make sure folders exist
for p in [DATA_DIR, LLM_DATA_DIR, RLHF_MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR        :", BASE_DIR.resolve())
print("MODEL_DIR (SFT) :", MODEL_DIR.resolve())
print("RLHF_MODEL_DIR  :", RLHF_MODEL_DIR.resolve())
print("SFT_DATA_PATH   :", SFT_DATA_PATH.resolve())

BASE_DIR        : /Users/imvenu/turing-ai-project
MODEL_DIR (SFT) : /Users/imvenu/turing-ai-project/models/sft_gpt2
RLHF_MODEL_DIR  : /Users/imvenu/turing-ai-project/models/rlhf_gpt2
SFT_DATA_PATH   : /Users/imvenu/turing-ai-project/data/sft/sales_llm_instructions.csv


## 🧱 Cell 3 — Load SFT data
👉 We expect columns like: ["instruction", "input", "output"] or ["instruction", "response"].
If your column names differ, just adjust in later cells.


In [10]:
# ==========================
# 1. Load instruction–response dataset
# ==========================

df = pd.read_csv(SFT_DATA_PATH)

print("Columns:", df.columns.tolist())
df.head()

Columns: ['instruction', 'output']


,instruction,output
0,Why do painkiller sales follow predictable wee...,Sales patterns are shaped by refill cycles and...
1,Analyze sales for R06 on 2014-04-06.,"On 2014-04-06, category R06 recorded 1.0 units..."
2,Analyze sales for M01AE on 2014-06-01.,"On 2014-06-01, category M01AE recorded 0.7 uni..."
3,Analyze sales for N02BE on 2014-03-24.,"On 2014-03-24, category N02BE recorded 23.6 un..."
4,Analyze sales for N05B on 2014-06-21.,"On 2014-06-21, category N05B recorded 12.0 uni..."


## 🧱 Cell 4 — Load base & SFT models

We’ll use:
	•	Reference model: your SFT model (this is what we want to further improve)
	•	Policy model: a copy of the SFT model that DPO will update

In [11]:
# ==========================
# 2. Load tokenizer & models
# ==========================  # base architecture model_name = "gpt2"

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load SFT model
ref_model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
ref_model.to(device)
ref_model.eval()

# Policy model (trainable copy)
policy_model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
policy_model.to(device)
policy_model.train()

print("Models loaded successfully!")

Models loaded successfully!


## 🧱 Cell 5 — Helper: generate answers from a model

We need a function to make the model answer an instruction.

In [12]:
import torch
import re

def ask_model(model, instruction, label="Model"):
    prompt = (
        "You are a pharmacy operations analyst.\n"
        "Give a concise, professional explanation.\n"
        "Focus only on patient behavior and clinic schedules.\n"
        "Do not include references, codes, or extra commentary.\n\n"
        f"Instruction: {instruction}\nAnswer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.25,
            top_p=0.65,
            repetition_penalty=1.7,
            no_repeat_ngram_size=6,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in text:
        text = text.split("Answer:", 1)[1].strip()

    # # CLEANUP
    # # REMOVE hallucinations + noise
    # text = re.sub(r"\(.*?\)", "", text)                  # remove brackets
    # text = re.sub(r"Recommendation.*", "", text)        # remove fake refs
    # text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)       # remove repeats
    # text = re.sub(r"\s+", " ", text).strip()

    # # FORCE 2 sentences max
    # sentences = text.split(".")
    # text = ". ".join(sentences[:2]).strip()

    # # Capitalize properly
    # text = text[:1].upper() + text[1:]
    # 🔥 STRONG CLEANUP

    # remove anything inside brackets
    text = re.sub(r"\(.*?\)", "", text)

    # remove 'Recommendation' phrases anywhere
    text = re.sub(r"Recommendation[^.]*", "", text, flags=re.IGNORECASE)

    # remove repeated words
    text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)

    # remove long generic fluff phrases
    text = re.sub(r"maximizing patient satisfaction.*", "", text, flags=re.IGNORECASE)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    # ✂️ FORCE SHORT OUTPUT (VERY IMPORTANT)
    text = text.split(".")[0].strip()

    # Capitalize
    text = text[:1].upper() + text[1:]
    
    print(f"\n🔹 {label} says:\n{text}\n")
    return text

In [13]:
print(ask_model.__code__.co_consts)

('You are a pharmacy operations analyst.\nGive a concise, professional explanation.\nFocus only on patient behavior and clinic schedules.\nDo not include references, codes, or extra commentary.\n\nInstruction: ', '\nAnswer:', 'pt', True, 256, ('return_tensors', 'truncation', 'max_length'), 'max_new_tokens', 'do_sample', 'temperature', 0.25, 'top_p', 0.65, 'repetition_penalty', 1.7, 'no_repeat_ngram_size', 'pad_token_id', 'eos_token_id', None, ('skip_special_tokens',), 'Answer:', '\\(.*?\\)', '', 'Recommendation[^.]*', ('flags',), '\\b(\\w+)\\s+\\1\\b', '\\1', 'maximizing patient satisfaction.*', '\\s+', ' ', '.', slice(None, 1, None), slice(1, None, None), '\n🔹 ', ' says:\n', '\n', ())


In [14]:
# ==========================
# 3. Helper: generation
# ==========================

def generate_answer(model, instruction, max_new_tokens=80, temperature=0.7, top_p=0.9):
    """
    Generate a text answer from a model for a single instruction.
    """
    model.eval()
    prompt = f"Instruction: {instruction}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.3,          # ↓ less randomness
            top_p=0.7,
            repetition_penalty=1.6,   # ↑ reduce repetition
            no_repeat_ngram_size=5,   # stronger control
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the prompt to return only the answer part
    if "Answer:" in text:
        text = text.split("Answer:", 1)[1].strip()
    return text

In [15]:
#import re

# remove weird patterns
#text = re.sub(r"\(.*?\)", "", text)   # remove brackets
#text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)  # remove repeated words
#text = re.sub(r"\s+", " ", text).strip()

## 🧱 Cell 6 — Build synthetic preference pairs

We’ll create:
	•	chosen: a better answer (from SFT or ground truth)
	•	rejected: a worse answer (shortened / no context / random)

This mimics human “thumbs up / thumbs down”.

In [16]:
# ==========================
# PREPARE df_pref for RLHF
# ==========================

from pathlib import Path
import pandas as pd

SFT_DATA_PATH = Path("../data/sft/sales_llm_instructions.csv")

df_pref = pd.read_csv(SFT_DATA_PATH)

# Column names
instr_col = "instruction"
out_col = "output"

# Optional: reduce size for faster training
df_pref = df_pref.sample(n=min(500, len(df_pref)), random_state=42).reset_index(drop=True)

print("Dataset loaded:", df_pref.shape)
df_pref.head()

Dataset loaded: (275, 2)


,instruction,output
0,Summarize the main contributors to weekly dema...,Patient pickup habits strongly influence daily...
1,Analyze sales for N02BA on 2014-05-21.,"On 2014-05-21, category N02BA recorded 9.5 uni..."
2,Analyze sales for M01AE on 2014-07-02.,"On 2014-07-02, category M01AE recorded 0.3 uni..."
3,Analyze sales for R06 on 2014-04-07.,"On 2014-04-07, category R06 recorded 1.0 units..."
4,Explain why weekends typically see higher phar...,Inventory should be adjusted ahead of predicta...


In [17]:
# ==========================
# 4. Build synthetic preference pairs
# ==========================
# Our simplified RLHF simulation:
#   - chosen: the "good" answer
#   - rejected: a "bad" answer
#
# We can choose "good" as the ground-truth output you wrote or generated.
# The "bad" answer is a noisy/truncated variant.


from tqdm import tqdm
import pandas as pd
import random

pref_rows = []

for _, row in tqdm(df_pref.iterrows(), total=len(df_pref), desc="Building preference pairs"):
    instr = row[instr_col]
    true_answer = str(row[out_col])

    # Good answer
    chosen = true_answer.strip()

    # Bad answer (improved RLHF signal)
    rejected_options = [
        "Sales are random and cannot be predicted.",
        "Increase inventory without considering patient demand patterns.",
        "Staffing decisions should not depend on patient flow.",
        "No clear trend can be observed from the data.",
        "All days have similar demand so no adjustments are needed.",
    ]

    if len(chosen.split()) > 15 and random.random() < 0.3:
        rejected = " ".join(chosen.split()[:5]) + " ..."
    else:
        rejected = random.choice(rejected_options)

    # Prompt
    prompt = f"Instruction: {instr}\nAnswer:"

    pref_rows.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    })

pref_df = pd.DataFrame(pref_rows)
pref_df.head()

Building preference pairs: 100%|███████████| 275/275 [00:00<00:00, 51181.82it/s]


,prompt,chosen,rejected
0,Instruction: Summarize the main contributors t...,Patient pickup habits strongly influence daily...,Sales are random and cannot be predicted.
1,Instruction: Analyze sales for N02BA on 2014-0...,"On 2014-05-21, category N02BA recorded 9.5 uni...",All days have similar demand so no adjustments...
2,Instruction: Analyze sales for M01AE on 2014-0...,"On 2014-07-02, category M01AE recorded 0.3 uni...","On 2014-07-02, category M01AE recorded ..."
3,Instruction: Analyze sales for R06 on 2014-04-...,"On 2014-04-07, category R06 recorded 1.0 units...","On 2014-04-07, category R06 recorded ..."
4,Instruction: Explain why weekends typically se...,Inventory should be adjusted ahead of predicta...,Increase inventory without considering patient...


## 🧱 Cell 7 — Create chosen vs rejected texts

In [18]:
# We will create three columns: prompt, chosen, rejected

pref_rows = []

for _, row in tqdm(df_pref.iterrows(), total=len(df_pref), desc="Building preference pairs"):
    instr = row[instr_col]
    true_answer = str(row[out_col])

    # Good answer: either ground truth or SFT model refinement
    # (to keep it simple, we'll just use true_answer)
    chosen = true_answer.strip()

    # Bad answer: truncated or noisy variant
    if len(chosen.split()) > 15:
        # Truncate to 5 words to simulate an incomplete / bad response
        rejected = " ".join(chosen.split()[:5]) + " ..."
    else:
        # If short, use a generic low-quality sentence
        rejected = "Data not available. Please check later."

    # Prompt text for the model
    prompt = f"Instruction: {instr}\nAnswer:"

    pref_rows.append(
        {
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
        }
    )

pref_df = pd.DataFrame(pref_rows)
pref_df.head()

Building preference pairs: 100%|███████████| 275/275 [00:00<00:00, 56763.46it/s]


,prompt,chosen,rejected
0,Instruction: Summarize the main contributors t...,Patient pickup habits strongly influence daily...,Data not available. Please check later.
1,Instruction: Analyze sales for N02BA on 2014-0...,"On 2014-05-21, category N02BA recorded 9.5 uni...","On 2014-05-21, category N02BA recorded ..."
2,Instruction: Analyze sales for M01AE on 2014-0...,"On 2014-07-02, category M01AE recorded 0.3 uni...","On 2014-07-02, category M01AE recorded ..."
3,Instruction: Analyze sales for R06 on 2014-04-...,"On 2014-04-07, category R06 recorded 1.0 units...","On 2014-04-07, category R06 recorded ..."
4,Instruction: Explain why weekends typically se...,Inventory should be adjusted ahead of predicta...,Data not available. Please check later.


## 🧱 Cell 8 — Convert to HuggingFace Dataset

In [19]:
from datasets import Dataset

# Create Dataset from pandas
pref_dataset = Dataset.from_pandas(pref_df)

pref_dataset = pref_dataset.shuffle(seed=42)

train_test = pref_dataset.train_test_split(test_size=0.1)

train_ds = train_test["train"]
eval_ds = train_test["test"]

print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))

# Split into train/validation (90/10)
split_dataset = pref_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_dataset["train"]
eval_ds = split_dataset["test"]

train_ds, eval_ds

Train size: 247
Eval size: 28


(Dataset({
     features: ['prompt', 'chosen', 'rejected'],
     num_rows: 247
 }),
 Dataset({
     features: ['prompt', 'chosen', 'rejected'],
     num_rows: 28
 }))

## 🧱 Cell 9 — DPO Training Config

trl.DPOTrainer requires:
	•	policy model
	•	reference model
	•	tokenizer
	•	dataset with prompt, chosen, rejected

In [20]:
# ==========================
# 5. Configure DPO training
# ==========================

# Hyperparameters are intentionally tiny so it can run on your MacBook
dpo_config = DPOConfig(
    output_dir=str(RLHF_MODEL_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1.0,     # start small; you can increase later
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=50,
    eval_strategy="epoch",
    warmup_ratio=0.1,
    beta=0.1,                 # DPO temperature; default 0.1
    max_length=256,
    max_prompt_length=128,
    remove_unused_columns=False,
    push_to_hub=False,
)

dpo_config

DPOConfig(output_dir='../models/rlhf_gpt2', overwrite_output_dir=False, do_train=False, do_eval=True, do_predict=False, eval_strategy=<IntervalStrategy.EPOCH: 'epoch'>, prediction_loss_only=False, per_device_train_batch_size=2, per_device_eval_batch_size=2, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=4, eval_accumulation_steps=None, eval_delay=0, torch_empty_cache_steps=None, learning_rate=5e-06, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=1.0, max_steps=-1, lr_scheduler_type=<SchedulerType.COSINE: 'cosine'>, lr_scheduler_kwargs=None, warmup_ratio=0.1, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir='../models/rlhf_gpt2/runs/Apr16_16-29-20_Venus-MacBook-Air.local', logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=10, logging_nan_inf_filter=True, save_strategy=<SaveStrategy.STEPS: 'ste

## 🧱 Cell 10 — Initialize DPOTrainer

⏳ This step will take a bit — but with the small dataset + tiny epochs it should be okay on MPS/CPU.

In [21]:
# ==========================
# 6. Initialize the DPOTrainer
# ==========================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_config,           # <- config already includes beta
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

print("DPOTrainer initialized ✅")

Extracting prompt in train dataset: 100%|█| 247/247 [00:00<00:00, 29581.21 examp
Applying chat template to train dataset: 100%|█| 247/247 [00:00<00:00, 42050.29 
Tokenizing train dataset: 100%|██████| 247/247 [00:00<00:00, 7630.28 examples/s]
Extracting prompt in eval dataset: 100%|█| 28/28 [00:00<00:00, 11843.54 examples
Applying chat template to eval dataset: 100%|█| 28/28 [00:00<00:00, 15863.91 exa
Tokenizing eval dataset: 100%|█████████| 28/28 [00:00<00:00, 5881.14 examples/s]

DPOTrainer initialized ✅


## 🧱 Cell 11 — Train RLHF-style model

⏳ This step will take a bit — but with the small dataset + tiny epochs it should be okay on MPS/CPU.

In [22]:
# ==========================
# 7. Train the policy model with DPO
# ==========================

train_result = dpo_trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.022300,0.017065,0.546110,-3.639175,1.000000,4.185285,-26.500734,-85.498466,-181.548416,-202.095596


/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=31, training_loss=0.16242358671321022, metrics={'train_runtime': 47.1146, 'train_samples_per_second': 5.243, 'train_steps_per_second': 0.658, 'total_flos': 0.0, 'train_loss': 0.16242358671321022, 'epoch': 1.0})

## 🧱 Cell 12 — Save model & tokenizer

In [23]:
# Save final RLHF-tuned model
RLHF_MODEL_DIR.mkdir(parents=True, exist_ok=True)

dpo_trainer.save_model(str(RLHF_MODEL_DIR))
tokenizer.save_pretrained(str(RLHF_MODEL_DIR))

print("RLHF-tuned model saved to:", RLHF_MODEL_DIR.resolve())

rlhf_model = GPT2LMHeadModel.from_pretrained(str(RLHF_MODEL_DIR)).to(device)
rlhf_model.eval()

RLHF-tuned model saved to: /Users/imvenu/turing-ai-project/models/rlhf_gpt2


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 🧱 Cell 13 — Helper: Ask the RLHF model vs SFT model

Now let’s compare answers:

In [24]:
print("RLHF model exists:", RLHF_MODEL_DIR.exists())

RLHF model exists: True


## 🧱 Cell 14 — Handy ask() for interactive use

You can now use the RLHF-tuned model like a little chatbot:

In [25]:
import torch

def ask_model(model, instruction, label="Model"):
    prompt = f"Instruction: {instruction}\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            top_p=0.9,
            temperature=0.6,
            repetition_penalty=1.4,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in text:
        text = text.split("Answer:", 1)[1].strip()

    print(f"\n🔹 {label} says:\n{text}\n")
    return text

In [26]:
def ask_rlhf(prompt):
   return ask_model(rlhf_model, prompt, label="RLHF Model")

#ask_rlhf("Summarize the pharmacy sales behavior for weekends and staffing suggestions.")

_ = ask_rlhf("Summarize the pharmacy sales behavior for weekends and staffing suggestions.")


🔹 RLHF Model says:
Provide predictable pickup schedules to align with clinic-driven weekly routines, maintain service quality while maximizing patient satisfaction through timely appointment appointments and scheduling consistency across clinics (see Recommendation R06).



In [27]:
print(RLHF_MODEL_DIR)

../models/rlhf_gpt2


In [28]:
print(rlhf_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)
